## Imports

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

##Configuration

In [0]:
CATALOG = "workspace"
SCHEMA = "final"

DQ_AUDIT_TABLE = f"{CATALOG}.{SCHEMA}.silver_data_quality_audit"
REJECT_TABLE = f"{CATALOG}.{SCHEMA}.silver_reject_records"
PIPELINE_AUDIT_TABLE = f"{CATALOG}.{SCHEMA}.silver_pipeline_audit"

## Data Quality Audit Table

In [0]:
spark.sql(f"""

CREATE TABLE IF NOT EXISTS {DQ_AUDIT_TABLE}
(
    table_name STRING,
    rule_name STRING,
    failed_count BIGINT,
    execution_timestamp TIMESTAMP
)

USING DELTA

""")

In [0]:
spark.table(DQ_AUDIT_TABLE).printSchema()

##Reject Records Table

In [0]:
spark.sql(f"""

CREATE TABLE IF NOT EXISTS {REJECT_TABLE}
(
    table_name STRING,
    rule_name STRING,
    reject_record STRING,
    execution_timestamp TIMESTAMP
)

USING DELTA

""")

In [0]:
spark.table(REJECT_TABLE).printSchema()

##Pipeline Audit Table

In [0]:
spark.sql(f"""

CREATE TABLE IF NOT EXISTS {PIPELINE_AUDIT_TABLE}
(
    table_name STRING,
    source_count BIGINT,
    target_count BIGINT,
    rejected_count BIGINT,
    execution_timestamp TIMESTAMP
)

USING DELTA

""")

In [0]:
spark.table(PIPELINE_AUDIT_TABLE).printSchema()

##Reusable Function: Log DQ Result

In [0]:
def log_dq_result(
        table_name,
        rule_name,
        failed_count):

    dq_df = spark.createDataFrame(
        [
            (
                table_name,
                rule_name,
                failed_count
            )
        ],
        [
            "table_name",
            "rule_name",
            "failed_count"
        ]
    ).withColumn(
        "execution_timestamp",
        current_timestamp()
    )

    (
        dq_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(DQ_AUDIT_TABLE)
    )

##Reusable Function: Log Reject Records

In [0]:
def log_reject_records(
        reject_df,
        table_name,
        rule_name):

    reject_output = (
        reject_df
        .select(
            to_json(
                struct("*")
            ).alias("reject_record")
        )
        .withColumn(
            "table_name",
            lit(table_name)
        )
        .withColumn(
            "rule_name",
            lit(rule_name)
        )
        .withColumn(
            "execution_timestamp",
            current_timestamp()
        )
    )

    (
        reject_output
        .select(
            "table_name",
            "rule_name",
            "reject_record",
            "execution_timestamp"
        )
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(REJECT_TABLE)
    )

##Reusable Function: Log Pipeline Audit

In [0]:
def log_pipeline_audit(
        table_name,
        source_count,
        target_count,
        rejected_count):

    audit_df = spark.createDataFrame(
        [
            (
                table_name,
                source_count,
                target_count,
                rejected_count
            )
        ],
        [
            "table_name",
            "source_count",
            "target_count",
            "rejected_count"
        ]
    ).withColumn(
        "execution_timestamp",
        current_timestamp()
    )

    (
        audit_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(
            PIPELINE_AUDIT_TABLE
        )
    )

##Example Usage

###Customer DQ

In [0]:
customer_df = spark.table(
    "workspace.final.silver_customer"
)

In [0]:
#Find bad records:
customer_reject_df = customer_df.filter(
    col("CustomerID").isNull()
)

In [0]:
#Count:
failed_count = customer_reject_df.count()

In [0]:
#Log result:
log_dq_result(
    table_name="silver_customer",
    rule_name="CustomerID_Not_Null",
    failed_count=failed_count
)

In [0]:
#Log rejects:
log_reject_records(
    customer_reject_df,
    "silver_customer",
    "CustomerID_Not_Null"
)

### Example Pipeline Audit

In [0]:
source_count = spark.table(
    "workspace.final.bronze_customer"
).count()

target_count = spark.table(
    "workspace.final.silver_customer"
).count()

rejected_count = (
    source_count -
    target_count
)

In [0]:
log_pipeline_audit(
    table_name="silver_customer",
    source_count=source_count,
    target_count=target_count,
    rejected_count=rejected_count
)

### View DQ Audit Table

In [0]:
display(
    spark.table(
        DQ_AUDIT_TABLE
    )
)

### View Reject Records

In [0]:
display(
    spark.table(
        REJECT_TABLE
    )
)

### View Pipeline Audit

In [0]:
display(
    spark.table(
        PIPELINE_AUDIT_TABLE
    )
)

Production Note

In an enterprise Databricks environment, these reusable functions
would typically be stored in a shared utility notebook and imported
using:

%run ../common/Data_Quality_Framework

This project keeps the framework in a dedicated notebook for
simplicity and portfolio demonstration purposes.